# 04 · Baseline Training

**Goals**
- Baseline segmentation model (U-Net, ResNet34 encoder), scored with local Panoptic Quality (`src/metrics.py`)
- Settle the question notebook 02 left to training: **which labels** (each / soft / vote)
- Tune the post-processing (threshold, minimum component area), compare with the annotator ceiling
- Error analysis (which kinds of mistakes the model makes), model size, training and end-to-end inference time

**Formulation.** Binary semantic segmentation (filament vs background) at 1024 px; instances are the
connected components (8-connectivity) of the thresholded probability map, upsampled to 2048 px and
restricted to the disk. Touching filaments therefore merge and gaps split a filament: these are the
main failure modes of this formulation, and this baseline measures how much they cost.

**Inputs come only from the previous notebooks:** `image_stats.csv` (01), `excluded_annotations.csv`,
`folds.csv` and the preprocessed images `cache/full_1024/` (02). Nothing is recomputed here.
- Locally: `artifacts/` and `cache/full_1024/` in the repo.
- On Kaggle: the CSVs come with the cloned repo (or an attached notebook output, which takes precedence);
  the preprocessed images come from an attached dataset of the `cache/full_1024/*.npy` files (any folder layout).

**Running.** Needs a GPU: on Kaggle, turn on GPU and Internet and use *Save & Run All*. Every run is appended
to `artifacts/baseline_runs.csv` and skipped if already there; download `artifacts/` and `checkpoints/` from
the Kaggle output afterwards. `COS711_SMOKE=1` runs the whole notebook on a few images on CPU, to check the pipeline.

In [ ]:
# --- Setup: works both locally and on Kaggle ---
import os, sys
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_KAGGLE:
    # Pull the shared src/ code from GitHub (requires Internet ON in notebook settings)
    if not Path("/kaggle/working/COS711-A2").exists():
        !git clone -q https://github.com/AryanMohanlall/COS711-A2.git /kaggle/working/COS711-A2
    ROOT = Path("/kaggle/working/COS711-A2")
elif Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

sys.path.insert(0, str(ROOT))

from src.data import get_data_dir
DATA_DIR = get_data_dir()
print("Kaggle:", IN_KAGGLE, "| data dir:", DATA_DIR)

try:
    import segmentation_models_pytorch
except ImportError:
    %pip install -q segmentation-models-pytorch

In [ ]:
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

from src.data import SITES, load_annotations, load_image
from src.preprocess import preprocess, find_disk, load_preprocessed, disk_mask
from src.augment import AugConfig, NO_AUG
from src.dataset import FilamentDataset, model_input
from src.model import build_model, BCEDiceLoss, model_size
from src.train import make_loader, fit, predict, evaluate, summarise
from src.postprocess import to_instances
from src.metrics import match_instances, pq_from_counts, error_types

SMOKE = os.environ.get("COS711_SMOKE") == "1"   # tiny CPU run: checks the pipeline, numbers meaningless
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

S = 1024                          # working resolution (notebook 01), = the cache notebook 02 built
FULL = 2048                       # evaluation resolution = original images
LR = 3e-4
VAL_FOLD = 0
RUN_CV = True                     # section 7: 4 more trainings (~50 min each on a T4)
THRESHOLDS = (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
MIN_AREAS = (0, 100, 200, 400, 600, 800, 1200, 1600)   # full-resolution px

if SMOKE:
    OUT_DIR = ROOT / "outputs" / "smoke"   # smoke results never land in artifacts/
    EPOCHS = 1
    EVAL_EVERY = 1
    BATCH = 2
    WORKERS = 0
    WEIGHTS = None
else:
    OUT_DIR = ROOT
    EPOCHS = 35
    EVAL_EVERY = 2                # validation PQ every 2 epochs; the best epoch is kept
    BATCH = 4
    WORKERS = min(4, os.cpu_count())
    WEIGHTS = "imagenet"

FIG_DIR = OUT_DIR / "figures" / "04"
RUN_DIR = OUT_DIR / "artifacts" / "runs"
CKPT_DIR = OUT_DIR / "checkpoints"
RUNS_CSV = OUT_DIR / "artifacts" / "baseline_runs.csv"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 4)


def previous_output(rel):
    """A file written by notebooks 01–02: an attached notebook output on Kaggle if there is one, else the repo copy."""
    candidates = []
    if IN_KAGGLE:
        candidates = sorted(Path("/kaggle/input").glob(f"**/{rel}"))
    candidates.append(ROOT / rel)
    for path in candidates:
        if path.exists():
            print(rel, "←", path)
            return path
    raise FileNotFoundError(f"{rel} not found: run notebooks 01–02 first")


def preprocessed_dir(probe):
    """Folder that holds full_<S>/ with the preprocessed .npy files of notebook 02.
    Locally that is the repo's cache/; on Kaggle, wherever an attached input holds the files."""
    if IN_KAGGLE:
        candidates = []
        for f in sorted(Path("/kaggle/input").glob(f"**/{probe}")):
            candidates.append(f.parent)
    else:
        candidates = [ROOT / "cache" / f"full_{S}"]

    found = None
    for folder in candidates:
        if (folder / probe).exists():
            found = folder
            break
    if found is None:
        message = f"preprocessed images ({probe}) not found: run notebook 02"
        if IN_KAGGLE:
            message += f", upload cache/full_{S} as a Kaggle dataset and attach it"
        raise FileNotFoundError(message)
    print(f"cache/full_{S}", "←", found)

    # load_preprocessed reads <cache_dir>/full_<S>/<obs>.npy, so a folder with another name is linked under /tmp
    if found.name != f"full_{S}":
        link = Path("/tmp/cos711_cache") / f"full_{S}"
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            link.symlink_to(found, target_is_directory=True)
        found = link
    return found.parent


IMAGE_STATS_CSV = previous_output("artifacts/image_stats.csv")            # notebook 01
EXCLUDED_CSV = previous_output("artifacts/excluded_annotations.csv")      # notebook 02
FOLDS_CSV = previous_output("artifacts/folds.csv")                        # notebook 02
first_obs = pd.read_csv(FOLDS_CSV).obs.iloc[0]
CACHE_DIR = preprocessed_dir(f"{first_obs}.npy")                         # notebook 02

if DEVICE.type == "cuda":
    print("device:", DEVICE, torch.cuda.get_device_name(), "| smoke:", SMOKE)
else:
    print("device:", DEVICE, "| smoke:", SMOKE)

## 1. Data

Cleaned annotations (notebook 02), the grouped, site-stratified folds (`artifacts/folds.csv`) and the
full-resolution disk geometry (`artifacts/image_stats.csv`). Fold 0 is the validation set for model
selection; folds 1–4 are only used for the optional cross-validation (section 7).

In [ ]:
images, anns = load_annotations(DATA_DIR)
excluded = pd.read_csv(EXCLUDED_CSV)
anns = anns[~anns.id.isin(excluded.ann_id)].reset_index(drop=True)
folds = pd.read_csv(FOLDS_CSV)
geo_full = pd.read_csv(IMAGE_STATS_CSV).set_index("obs")[["cx", "cy", "radius"]]

if SMOKE:   # a few observations per fold
    folds = folds.groupby("fold").head(3).reset_index(drop=True)


def split(fold):
    """(train observations, validation observations) when `fold` is the validation fold."""
    train = folds.loc[folds.fold != fold, "obs"].tolist()
    val = folds.loc[folds.fold == fold, "obs"].tolist()
    return train, val


train_obs, val_obs = split(VAL_FOLD)
print(f"train {len(train_obs)} obs / val {len(val_obs)} obs")

# Observations per site in train and validation
parts = folds.copy()
parts["part"] = np.where(parts.fold == VAL_FOLD, "val", "train")
parts.pivot_table(index="part", columns="site", values="obs", aggfunc="count")

In [ ]:
# Preprocessed inputs (mode "full") from notebook 02: every observation must be there, nothing is rebuilt
paths = images.drop_duplicates("obs").set_index("obs").path
cache_folder = CACHE_DIR / f"full_{S}"
missing = []
for o in folds.obs:
    if not (cache_folder / f"{Path(paths[o]).stem}.npy").exists():
        missing.append(o)
if missing:
    raise FileNotFoundError(f"{len(missing)} observations missing from {cache_folder}, e.g. {missing[:3]}: "
                            "rerun notebook 02 / re-upload the dataset")
print(f"{len(folds)} preprocessed observations in {cache_folder}")

### Input normalisation and class imbalance

The network input is `(x − centre) / scale` with a **fixed** centre and scale (median and robust std of
on-disk pixels over the training images). Per-image standardisation would undo the contrast augmentation
of notebook 03. The filament fraction of disk pixels shows how imbalanced the problem is.

In [ ]:
probe = FilamentDataset(train_obs, images, anns, geo_full, size=S, cache_dir=CACHE_DIR)
step = max(1, len(probe.obs) // 100)
sample = probe.obs[::step]   # about 100 training images, evenly spread

values = []
fg = []
for o in sample:
    cx, cy, r = geo_full.loc[o]
    g = (cx * S / FULL, cy * S / FULL, r * S / FULL)   # disk geometry at working resolution
    x = load_preprocessed(paths[o], (cx, cy, r), S, cache_dir=CACHE_DIR)
    on_disk_pixels = x[disk_mask(x.shape, g[0], g[1], g[2], 0.985)]
    values.append(on_disk_pixels[::7])   # every 7th pixel is plenty for a median
    on_disk = disk_mask((S, S), g[0], g[1], g[2])
    for m in probe.masks(o):
        fg.append((m[on_disk] > 0).mean())
values = np.concatenate(values)
fg = np.array(fg)

# Fixed input normalisation: median and robust std of on-disk pixels
med = float(np.median(values))
scale = float(1.4826 * np.median(np.abs(values - med)))
NORM = (med, scale)
print("normalisation (centre, scale):", (round(med, 4), round(scale, 4)))
mean_pct = round(100 * fg.mean(), 2)
median_pct = round(100 * np.median(fg), 2)
max_pct = round(100 * fg.max(), 2)
ratio = int(round(1 / fg.mean()))
print(f"filament pixels per annotation set: {mean_pct}% of the disk "
      f"(median {median_pct}%, max {max_pct}%) → about 1 : {ratio}")

## 2. Model, loss and training

| Choice | Setting | Why |
|---|---|---|
| Architecture | U-Net, ResNet34 encoder (ImageNet weights, stem collapsed to 1 channel), `segmentation_models_pytorch` | Skip connections keep the 3–6 px detail of thin filaments at full output resolution; a pretrained encoder converges fast on ~560 training images; filaments rarely touch (2.9% have a neighbour within 10 px), so connected components are a reasonable first instance step |
| Loss | 0.5 · BCE + 0.5 · soft Dice (per image) | Filaments are ~0.6% of the disk (above): Dice does not reward predicting all background; BCE gives stable per-pixel gradients |
| Optimiser | AdamW, lr 3e-4, weight decay 1e-4, 1 epoch warmup + cosine decay, fp16 autocast | Standard fine-tuning recipe |
| Batch / epochs | 4 × 1024² / 35 epochs, validation PQ every 2 epochs, best epoch kept | Fits a 16 GB Kaggle GPU |
| Augmentation | `AugConfig()` from notebook 03 | Calibrated against real and test images |
| Post-processing | threshold, disk mask, connected components, drop components < `min_area` (full-res px) | Simplest way from pixels to instances; threshold and `min_area` are tuned on the validation fold |

PQ is computed at 2048 px against **every annotation set** of each validation observation (counts pooled),
the same protocol as the annotator comparisons in notebooks 01–02.

In [ ]:
def datasets(label, fold, seed):
    """Training set (with augmentation) and validation set (without) for one fold."""
    tr_obs, va_obs = split(fold)
    train = FilamentDataset(tr_obs, images=images, anns=anns, geo_full=geo_full, label=label, size=S,
                            norm=NORM, cache_dir=CACHE_DIR, aug=AugConfig(), seed=seed)
    val = FilamentDataset(va_obs, images=images, anns=anns, geo_full=geo_full, label=label, size=S,
                          norm=NORM, cache_dir=CACHE_DIR, aug=NO_AUG)
    return train, val


def scorer(va):
    """Validation helpers: a function giving the full-resolution masks of every annotation set,
    and the full-resolution disk of every observation."""
    def gt_fn(o):
        return va.masks(o, FULL)

    disks = {}
    for o in va.obs:
        disks[o] = va.geo[o]
    return gt_fn, disks


def run(name, label="each", fold=VAL_FOLD, seed=0, epochs=EPOCHS):
    """Train one model, tune the post-processing on its validation fold, save everything, return a summary row."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    tr, va = datasets(label, fold, seed)
    gt_fn, disks = scorer(va)
    va_dl = make_loader(va, 2 * BATCH, train=False, workers=WORKERS)
    timer = {"eval_s": 0.0}   # time spent on validation, excluded from the training time

    def eval_fn(m):
        # Validation PQ at threshold 0.5, no size filter
        t0 = time.time()
        counts = evaluate(predict(m, va_dl), gt_fn, disks)
        pq = summarise(counts).pq.iloc[0]
        timer["eval_s"] += time.time() - t0
        return pq

    def log_line(text):
        print(f"[{name}] {text}")

    model = build_model(weights=WEIGHTS).to(DEVICE)
    t0 = time.time()
    train_dl = make_loader(tr, BATCH, train=True, workers=WORKERS)
    hist = fit(model, train_dl, BCEDiceLoss(), epochs, LR, eval_fn=eval_fn, eval_every=EVAL_EVERY, log=log_line)
    train_min = (time.time() - t0 - timer["eval_s"]) / 60

    # Tune threshold and min_area on the validation fold
    counts = evaluate(predict(model, va_dl), gt_fn, disks, THRESHOLDS, MIN_AREAS)
    summ = summarise(counts)
    t_best, a_best = summ.pq.idxmax()

    torch.save(model.state_dict(), CKPT_DIR / f"{name}.pt")
    hist.to_csv(RUN_DIR / f"{name}_history.csv", index=False)
    counts.to_csv(RUN_DIR / f"{name}_counts.csv", index=False)

    best_epoch = int(hist.loc[hist.val_pq.idxmax(), "epoch"])
    row = {"name": name, "label": label, "fold": fold, "seed": seed, "epochs": epochs, "size": S,
           "best_epoch": best_epoch, "train_min": train_min,
           "pq_default": summ.loc[(0.5, 0), "pq"], "threshold": t_best, "min_area": a_best,
           "pq": summ.loc[(t_best, a_best), "pq"], "sq": summ.loc[(t_best, a_best), "sq"],
           "rq": summ.loc[(t_best, a_best), "rq"]}
    write_header = not RUNS_CSV.exists()
    pd.DataFrame([row]).to_csv(RUNS_CSV, mode="a", header=write_header, index=False)

    del model
    torch.cuda.empty_cache()
    return row


def run_all(experiments):
    """Run every experiment not already in baseline_runs.csv; return their result rows."""
    if RUNS_CSV.exists():
        done = set(pd.read_csv(RUNS_CSV).name)
    else:
        done = set()

    names = []
    for e in experiments:
        names.append(e["name"])
        if e["name"] in done:
            print("skip (done):", e["name"])
            continue
        fold = VAL_FOLD
        if "fold" in e:
            fold = e["fold"]
        run(e["name"], label=e["label"], fold=fold)

    runs = pd.read_csv(RUNS_CSV).drop_duplicates("name", keep="last")
    return runs[runs.name.isin(names)].set_index("name")

## 3. Which training labels?

Notebook 02's leave-one-out test found that, as a *label*, combining annotators predicts a held-out annotator
no better than a single annotator does (PQ 0.360 union / 0.354 intersection vs 0.357). A network is different:
trained on **each** (a random annotation set per epoch) it sees conflicting targets but may learn the consensus
itself; **soft** (fraction of annotators per pixel) gives it that consensus directly; **vote** is a consistent
hard label. `union` is left out: it merged 173 pairs of separate filaments in notebook 02, which PQ punishes.
One model per label type, same seed, same fold. `pq_default` is at threshold 0.5 / no size filter; `pq` is
after tuning both.

In [ ]:
LABEL_EXPERIMENTS = []
for lab in ("each", "soft", "vote"):
    LABEL_EXPERIMENTS.append({"name": f"unet_r34_{lab}", "label": lab})
label_runs = run_all(LABEL_EXPERIMENTS)
label_runs[["label", "best_epoch", "train_min", "pq_default", "threshold", "min_area", "pq", "sq", "rq"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for name in label_runs.index:
    h = pd.read_csv(RUN_DIR / f"{name}_history.csv")
    axes[0].plot(h.epoch, h.loss, label=name)
    evaluated = h.dropna(subset=["val_pq"])   # validation PQ only exists for some epochs
    axes[1].plot(evaluated.epoch, evaluated.val_pq, marker=".", label=name)
axes[0].set(xlabel="epoch", ylabel="train loss", title="Training loss")
axes[1].set(xlabel="epoch", ylabel="val PQ (t = 0.5, no size filter)", title="Validation PQ")
axes[1].legend(fontsize=7)
plt.tight_layout()
savefig("training_curves")
plt.show()

## 4. Post-processing of the best model

In [ ]:
BEST = label_runs.pq.idxmax()
best = label_runs.loc[BEST]
T = float(best.threshold)
A = int(best.min_area)
print(f"best: {BEST}  (threshold {T}, min_area {A}, PQ {round(best.pq, 3)})")


def load_model(name):
    m = build_model(weights=None).to(DEVICE)
    m.load_state_dict(torch.load(CKPT_DIR / f"{name}.pt", map_location=DEVICE))
    return m.eval()


model = load_model(BEST)
_, va = datasets(best.label, VAL_FOLD, 0)
gt_fn, disks = scorer(va)
probs = predict(model, make_loader(va, 2 * BATCH, train=False, workers=WORKERS))
counts = pd.read_csv(RUN_DIR / f"{BEST}_counts.csv")

# PQ for every (threshold, min_area) pair
grid = summarise(counts).pq.unstack("min_area")
fig, ax = plt.subplots(figsize=(6, 3.5))
im = ax.imshow(grid.values, cmap="viridis", aspect="auto")
ax.set(xticks=range(grid.shape[1]), xticklabels=grid.columns, yticks=range(grid.shape[0]), yticklabels=grid.index,
       xlabel="min_area (full-res px)", ylabel="threshold", title=f"Validation PQ, {BEST}")
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        ax.text(j, i, str(round(grid.values[i, j], 3)), ha="center", va="center", fontsize=7, color="w")
plt.colorbar(im)
plt.tight_layout()
savefig("postprocess_grid")
plt.show()

## 5. Analysis on the validation fold

### Against the annotator ceiling

The annotators themselves disagree (notebook 01). Inter-annotator PQ on the validation observations
with ≥ 2 sets (every ordered pair of sets) is a realistic ceiling; the model is scored on the same
observations for a like-for-like comparison.

In [ ]:
# Validation observations with more than one annotation set
multi = []
for o in va.obs:
    if len(va.set_ids[o]) > 1:
        multi.append(o)

# Every annotator scored against every other annotator of the same observation
c = np.zeros(4)
for o in tqdm(multi, desc="annotator pairs"):
    ms = gt_fn(o)
    for i in range(len(ms)):
        for j in range(len(ms)):
            if i != j:
                m = match_instances(ms[i], ms[j])
                c += (m["tp"], m["fp"], m["fn"], m["iou_sum"])
ceiling = pq_from_counts(int(c[0]), int(c[1]), int(c[2]), c[3])

at_best = counts[(counts.threshold == T) & (counts.min_area == A)]
model_multi = summarise(at_best[at_best.obs.isin(multi)]).iloc[0]
comparison = pd.DataFrame({"annotator vs annotator": ceiling, "model vs annotators": model_multi})
comparison.loc[["pq", "sq", "rq"]].T

In [ ]:
per_img = summarise(at_best, by=["obs"]).join(folds.set_index("obs")[["site", "degraded"]])

groups = {}
for site, g in per_img.groupby("site"):
    groups[site] = summarise(at_best[at_best.obs.isin(g.index)]).iloc[0]
for is_degraded, g in per_img.groupby("degraded"):
    if is_degraded:
        key = "degraded"
    else:
        key = "clean"
    groups[key] = summarise(at_best[at_best.obs.isin(g.index)]).iloc[0]
by_group = pd.DataFrame(groups).T
display(by_group[["pq", "sq", "rq", "tp", "fp", "fn"]])

### What kind of errors?

Every unmatched ground-truth filament is **missed** (no predicted pixel on it), **merged** (its prediction
also covers another filament), **split** (covered by several predicted pieces) or **boundary** (one
prediction, IoU ≤ 0.5). Unmatched predictions on no filament are **spurious**. Merged and split are what
better instance separation could fix.

In [ ]:
err = []
for o in tqdm(va.obs, desc="error types"):
    pred = to_instances(probs[o], T, A, disks[o])
    for gt in gt_fn(o):
        row = {"obs": o}
        row.update(error_types(gt, pred))
        err.append(row)
err = pd.DataFrame(err)
tot = err.drop(columns="obs").sum()
fates = ["matched", "missed", "merged", "split", "boundary"]
gt_total = tot[fates].sum()
print((100 * tot[fates] / gt_total).round(1).to_string())
per_set = round(tot.spurious / len(err), 2)
print(f"spurious predictions: {int(tot.spurious)} ({per_set} per annotation set)")

fig, ax = plt.subplots(figsize=(7, 3))
shares = 100 * tot.drop("spurious") / gt_total
shares.plot.bar(ax=ax, color=["tab:green", "tab:gray", "tab:red", "tab:orange", "tab:blue"])
ax.set(ylabel="% of GT filaments", title="Fate of ground-truth filaments (validation)")
plt.xticks(rotation=0)
plt.tight_layout()
savefig("error_types")
plt.show()

In [ ]:
def show_case(ax, o, title):
    x = load_preprocessed(paths[o], disks[o], S, cache_dir=CACHE_DIR)
    pred = to_instances(probs[o], T, A, disks[o]).astype(np.float32)
    pred = cv2.resize(pred, (S, S), interpolation=cv2.INTER_NEAREST)
    gt = va.masks(o)[0]
    ax.imshow(x, cmap="gray", vmin=0.8, vmax=1.2)
    ax.contour((gt > 0).astype(float), levels=[0.5], colors="cyan", linewidths=0.6)
    ax.contour((pred > 0).astype(float), levels=[0.5], colors="red", linewidths=0.6)
    ax.set_title(title, fontsize=9)
    ax.set_axis_off()


ranked = per_img[per_img.tp + per_img.fn > 0].sort_values("pq")
best_three = list(ranked.index[-3:])
best_three.reverse()   # best first
worst_three = list(ranked.index[:3])
cases = best_three + worst_three

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i in range(len(cases)):
    o = cases[i]
    r = per_img.loc[o]
    place = SITES[r.site]
    if r.degraded:
        place = place + ", degraded"
    title = (f"{o} ({place})\nPQ {round(r.pq, 2)}  TP {int(r.tp)}  FP {int(r.fp)}  FN {int(r.fn)}")
    show_case(axes.flat[i], o, title)
fig.suptitle("Best (top) and worst (bottom) validation images. Cyan: annotation set 1, red: prediction", y=0.99)
plt.tight_layout()
savefig("best_worst_val")
plt.show()

## 6. Size, training time and end-to-end inference time

End to end means from the JPEG on disk to the instance map: load, disk detection, preprocessing,
network, post-processing. No cache is used.

In [ ]:
def end_to_end(path):
    """JPEG on disk → instance map, with the time of each step."""
    t = {}

    t0 = time.perf_counter()
    img = load_image(path)
    t["load"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    disk = find_disk(img)
    x = cv2.resize(preprocess(img, "full", disk), (S, S), interpolation=cv2.INTER_AREA)
    disk_small = (disk[0] * S / FULL, disk[1] * S / FULL, disk[2] * S / FULL)
    x = model_input(x, disk_small, NORM, 1.0)
    t["preprocess"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    batch = torch.from_numpy(x).reshape(1, 1, S, S).to(DEVICE)   # a batch of one single-channel image
    with torch.no_grad(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        logits = model(batch)
    p = torch.sigmoid(logits.float())[0, 0].cpu().numpy()
    t["network"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    inst = to_instances(p, T, A, disk)
    t["postprocess"] = time.perf_counter() - t0
    return inst, t


end_to_end(paths[va.obs[0]])   # warm-up (cuDNN autotuning, lazy init)
timing_rows = []
for o in va.obs[:30]:
    inst, t = end_to_end(paths[o])
    timing_rows.append(t)
timing = pd.DataFrame(timing_rows)
timing["total"] = timing.sum(axis=1)

size = model_size(model)
if DEVICE.type == "cuda":
    device_name = torch.cuda.get_device_name()
else:
    device_name = "cpu"
summary = {
    "model": BEST,
    "params (M)": round(size["params_M"], 2),
    "weights (MB, fp32)": round(size["size_MB"], 1),
    "training time (min, excl. validation)": round(float(best.train_min), 1),
    "inference ms / image (end to end)": round(1000 * timing.total.mean(), 1),
    "inference for 180 test images (s)": round(180 * timing.total.mean(), 1),
    "device": device_name,
}
display(pd.Series(summary).to_frame("value"))
(1000 * timing.mean()).round(1).to_frame("ms / image")

## 7. Cross-validation of the best configuration (`RUN_CV`)

The best label type retrained on folds 1–4 (fold 0 is the run above), with the threshold and `min_area`
**fixed** to the values tuned on fold 0, so folds 1–4 are honest held-out estimates. Fold 0's score is
optimistic: the epoch, threshold and `min_area` were all selected on it.

In [ ]:
if RUN_CV:
    CV_EXPERIMENTS = []
    for f in sorted(folds.fold.unique()):
        if f != VAL_FOLD:
            CV_EXPERIMENTS.append({"name": f"{BEST}_fold{f}", "label": best.label, "fold": f})
    run_all(CV_EXPERIMENTS)

    # Score every fold with the threshold and min_area tuned on fold 0
    fold_runs = [(BEST, VAL_FOLD)]
    for e in CV_EXPERIMENTS:
        fold_runs.append((e["name"], e["fold"]))
    cv = []
    for name, f in fold_runs:
        c = pd.read_csv(RUN_DIR / f"{name}_counts.csv")
        c = c[(c.threshold == T) & (c.min_area == A)]
        s = summarise(c).iloc[0]
        cv.append({"fold": f, "pq": s.pq, "sq": s.sq, "rq": s.rq})
    cv = pd.DataFrame(cv).set_index("fold")
    held_out = cv.drop(index=VAL_FOLD)
    pq_text = f"{round(held_out.pq.mean(), 3)} ± {round(held_out.pq.std(), 3)}"
    sq_text = f"{round(held_out.sq.mean(), 3)} ± {round(held_out.sq.std(), 3)}"
    rq_text = f"{round(held_out.rq.mean(), 3)} ± {round(held_out.rq.std(), 3)}"
    print(f"PQ folds 1–4: {pq_text}   (SQ {sq_text}, RQ {rq_text})")
    display(cv)

## Summary: Panoptic Quality

PQ on the validation fold (fold 0), pooled over every annotation set of every validation observation.

In [ ]:
pq_table = label_runs[["pq_default", "pq"]].rename(columns={
    "pq_default": "PQ (threshold 0.5, no size filter)",
    "pq": "PQ (tuned threshold and min_area)",
})
display(pq_table)

print(f"Best: {BEST}, PQ {round(best.pq, 3)} (threshold {T}, min_area {A})")
print(f"Observations with several annotation sets: model PQ {round(model_multi.pq, 3)} "
      f"vs annotator-vs-annotator PQ {round(ceiling['pq'], 3)}")
if RUN_CV:
    print(f"Cross-validated PQ, folds 1–4: {round(held_out.pq.mean(), 3)} ± {round(held_out.pq.std(), 3)}")